# Restaurant & Café Sales Trend Analysis
## Phase 1 — Cleaning

**Author:** Carl Mark Sibal

**Tools:** Python, Pandas

**Dataset Sources:** Balaji Fast Food Sales (India) & Dirty Café Sales (Canada)

---

## Project Overview
This project analyzes sales data **(synthetic)** from two food businesses — a Canadian café and an Indian fast food restaurant — using synthetic data sourced from Kaggle. The goal is to uncover revenue trends, top-performing items, and seasonal patterns, and to develop data-driven recommendations for sales and marketing improvement across seasons.

The analysis spans 4 phases:
- **Phase 1** — Data Cleaning <<--
- **Phase 2** — Exploratory Data Analysis 
- **Phase 3** — Combined Analysis & Weather API Integration
- **Phase 4** — Power BI Dashboard 

## Phase Overview
The cleaning part of this project will be heavily focused on the cafe dataset since it is the one containing much more nulls and ambiguous values.
The process will be done in stages to deal with nuances with the data. 




# Importing pandas and the dfs
------------------

In [18]:
import numpy as np
import pandas as pd
# the restaurant data
df = pd.read_csv(r"C:\Users\rougs\Desktop\data cleaning project\Balaji Fast Food Sales.csv")
# the cafe data
df2 = pd.read_csv(r"C:\Users\rougs\Desktop\data cleaning project\dirty_cafe_sales.csv", names=["Transaction_ID", "Item", "Quantity", "Price", "Total_spent", "Payment Method", "Order method", "date"])

## Checking for shape and general info about the dfs
-------------------------

In [19]:
# restaurant data
print('//Restaurant//')
print(df.describe())
print('\n','Cafe')
print(df2.columns)

//Restaurant//
          order_id   item_price     quantity  transaction_amount
count  1000.000000  1000.000000  1000.000000         1000.000000
mean    500.500000    33.315000     8.162000          275.230000
std     288.819436    14.921744     4.413075          204.402979
min       1.000000    20.000000     1.000000           20.000000
25%     250.750000    20.000000     4.000000          120.000000
50%     500.500000    25.000000     8.000000          240.000000
75%     750.250000    50.000000    12.000000          360.000000
max    1000.000000    60.000000    15.000000          900.000000

 Cafe
Index(['Transaction_ID', 'Item', 'Quantity', 'Price', 'Total_spent',
       'Payment Method', 'Order method', 'date'],
      dtype='object')


In [20]:
# checking for duplicates

dupes1 = df['order_id'].duplicated().sum()
dupes2 = df2['Transaction_ID'].duplicated().sum()

print(dupes1,dupes2)

0 0


## Casting and Standardizations
---------------------
The date and transaction_type columns from the Balaji restaurant is the only column in the dataset that requires standardization since the dataset is already sort of cleaned. While the cafe dataset requires rounds of imputation in multiple columns along with standardization.



In [21]:
#changing dates to consistent format
df['date'] = pd.to_datetime(df['date'], format='mixed')
df2['date'] = pd.to_datetime(df2['date'], format='mixed')

In [22]:
#Standardizing Payment Method and transaction_type columns
df['transaction_type'] = df['transaction_type'].fillna('Unknown')
df2['Payment Method'] = df2['Payment Method'].replace(['UNKNOWN','ERROR'], np.nan)

In [23]:
df2['Payment Method'] = df2['Payment Method'].fillna('Unknown')
print(
    df['transaction_type'].unique()
      ,df2['Payment Method'].unique()
) # checking

['Unknown' 'Cash' 'Online'] ['Credit Card' 'Cash' 'Unknown' 'Digital Wallet']


## Item standardization

------------------------------------------

After this process it is expected for the total number of nulls in the item column to increase since 'UNKNOWN' and 'ERROR' are not nulls, they add to the count of valid values in the column.

In [24]:
#Standardizing unknowns and errors
df2['Item'] = df2['Item'].replace(['UNKNOWN','ERROR'], np.nan)
print(
    df2['Item'].unique()
)
print(
    df2.isnull().sum()
)

['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' nan 'Sandwich' 'Juice' 'Tea']
Transaction_ID       0
Item               969
Quantity           479
Price              533
Total_spent        502
Payment Method       0
Order method      3265
date               460
dtype: int64


## Rounds of imputations for df2 Quantity, Price, Item and Total_spent column 
-------------------
Missing values in these four columns needs to be derived from one another. After the new values are added to their respective columns, another round of imputations will be done to take into account the new values that we get. 

But before moving on let's take a quick look at the null values


**DF2(Cafe)**
|Column|Nulls|
|------|-----|
|Transaction_ID |      0|
|Item             |  333|
|Quantity         |  479|
|Price             | 533|
|Total_spent       | 502|
|Payment Method    |2579|
|Order method     | 3265|
|date              | 460|

Missing values from Payment Method, Order Method and date cannot be derived and no data is readily available for us to use in place of the nulls so they will be left nulled to maintain data integrity and validity of any further analysis.


# First round
-------------------------
The values for each of these columns will be derived from this formula **Quantity = Total_spent/Price** and will be recycled and adjusted depending on which column is being derived. 

In [25]:
#Imputing df2['Item'] nulls with values that matches the unambiguous price
# first the table
reverse_lookup = {
    2: 'Coffee',
    1.5: 'Tea',
    5: 'Salad',
    1: 'Cookie'
}

## Missing Items

Note that if the missing item is sharing the same price with another item then it will not be imputed to avoid misinterpretations in any future business recommendations. However its Quantity, Price and Total_spent column will still be derived whenever possible to contribute to the completeness of the data. 

## Item Imputations
---------------------------------------


In [26]:

df2['Item'] = np.where(
    (df2['Item'].isnull()) &(df2['Price'].notnull()) & (~df2['Price'].isin([3, 4])),
           df2['Price'].map(reverse_lookup), 
            df2['Item']
)
print(
    df2['Item'].isnull().sum()
)


501


### Total_spent

---------------------------------------

In [27]:

df2['Total_spent'] = np.where(
    (df2['Total_spent'].isnull()) & (df2['Quantity'].notnull()) & (df2['Price'].notnull()),
    df2['Quantity'] * df2['Price'],
    df2['Total_spent']
)

#### Resulting nulls in Total_spent after imputations

----------------------------

In [28]:
# Total nulls from Total_spent column after imputation
print(
    df2['Total_spent'].isnull().sum()
) 
# Original count of nulls were 502 for this column

40


# Conditional imputations (2nd round)

----------------------------
After the initial imputations, columns Item, Quantity, Price and Total_spent would now have a reduced null count and would have new values that can be further used to derive more values from all these 4 columns. 

In [29]:
# Deriving Quantity and Price via conditional imputation
# same thing as before really
df2['Quantity'] = np.where(
    (df2['Quantity'].isnull()) & (df2['Total_spent'].notnull()) & (df2['Price'].notnull()),
    df2['Total_spent'] / df2['Price'],
    df2['Quantity']
)
df2['Price'] = np.where(
    (df2['Price'].isnull()) & (df2['Total_spent'].notnull()) & (df2['Quantity'].notnull()),
    df2['Total_spent'] / df2['Quantity'],
    df2['Price']
)

In [30]:
# Total nulls from Quantity and Price 
print(
    df2['Quantity'].isnull().sum()
)
print(
    df2['Price'].isnull().sum()
)
# Quantity and Price nulls were 479 and 533 consecutively
print(
    df2.isnull().sum()
)

38
38
Transaction_ID       0
Item               501
Quantity            38
Price               38
Total_spent         40
Payment Method       0
Order method      3265
date               460
dtype: int64


## Adjustments
---------------------------------
Total_spent null count cannot be further reduced from 40 
even after the conditional imputation of Quantity and Price
which means those 2 total_spent rows must be missing either a quantity or a price
| Column | Unrecoverable |
| ----------------- | -- |
| Total_spent  | 40 |
| Quantity     | 38 |
| Price        | 38 |

However with the help of a price_lookup from kaggle this might change


In [31]:
# Further reducing Item nulls by imputing given values from available pricing table
price_lookup = {
    'Coffee': 2,
    'Tea': 1.5,
    'Sandwich': 4,
    'Salad' : 5,
    'Cake' : 3,
    'Cookie' : 1 ,
    'Smoothie' : 4,
    'Juice' : 3,
}

## Price imputation logic 
-------------------------
Logic will be
If **Price is null** AND **Item** is not null THEN fill with matching price_lookup **item**
 **ELSE** keep existing **Price**

In [32]:
# Price imputations from table
df2['Price'] = np.where(
    (df2['Price'].isnull()) & (df2['Item'].notnull()),
    df2['Item'].map(price_lookup),
    df2['Price']  
)

In [33]:
# Price rows with nulls are now down to 6
# Re-run of conditional imputations to possibly recover more data 

df2['Quantity'] = np.where(
    (df2['Quantity'].isnull()) & (df2['Total_spent'].notnull()) & (df2['Price'].notnull()),
    df2['Total_spent'] / df2['Price'],
    df2['Quantity']
)
print(df2['Quantity'].isnull().sum())


23


In [34]:
df2['Total_spent'] = np.where(
    (df2['Total_spent'].isnull()) & (df2['Quantity'].notnull()) & (df2['Price'].notnull()),
    df2['Quantity'] * df2['Price'],
    df2['Total_spent']
)
df2['Price'] = np.where(
    (df2['Price'].isnull()) & (df2['Total_spent'].notnull()) & (df2['Quantity'].notnull()),
    df2['Total_spent'] / df2['Quantity'],
    df2['Price']
)
print(df2['Total_spent'].isnull().sum())
print(df2['Price'].isnull().sum())


23
6


After price-lookup re-run we bring down the total of total_spent nulls to 23 
| Column | Unrecoverable |
| ----------------- | -- |
| Total_spent  | 23| |
| Quantity     | 38 |
| Price        | 38 |


## Imputations AND Standardizations conclusion
-----------------------------------
After the rounds of imputation, nulls that are left can no longer be derived since they either are ambiguous or simply just unsafe to derive because one or more values are missing and cannot be calculated with a formula. 

Here is a summary of the cleaning: 
**DF2(Cafe)**
|Column|Nulls|Nulls(AFTER)|
|------|-----|-----------|
|Transaction_ID |      0| 0|
|Item             |  333|501|
|Quantity         |  479|23|
|Price             | 533|6|
|Total_spent       | 502|23|
|Payment Method    |2579|2579|
|Order method     | 3265|3265|
|date              | 460|460|

Notice that nulls from Item has increased after derivations. This is because the column had values like 'ERROR' and 'UNKNOWN' that were both standardized and left nulled. The missing 333 values from the beginning of the phase were not taking these 2 values into account before we ran the sum of all nulls.

## Flagging
----------------------


A flag column will be added to label columns with missing dates and also distinguish which rows are ready to be analyzed and which needs to be analyzed separately. 

In [35]:
df2['Flags'] = np.where(
    df2['date'].isnull(), 
     'Missing date',
     'Good'
    )
df2['Flags'].value_counts()


Flags
Good            9540
Missing date     460
Name: count, dtype: int64

### Saving unusable data in a separate 

------------------------------------

In [36]:
quarantine = df2[df2['Item'].isnull()]
quarantine.to_csv(r'C:\Users\rougs\Desktop\data cleaning project\removed_records.csv', index=False)

In [37]:
# removing quarantine rows from df2
df2 = df2[~df2['Item'].isnull()]


In [38]:
# checking for shape of old and new df2 and the quanrantine df 
print(df2.shape)
print(len(quarantine))
print(quarantine.shape)

(9499, 9)
501
(501, 9)


In [39]:
#aliasing :) 
cafe_sales_clean = df2
fastfood_sales_clean = df


In [40]:
# both to new csv
# and with that, cleaning is done. To EDA
cafe_sales_clean.to_csv(r'C:\Users\rougs\Desktop\data cleaning project\cafeclean.csv',index=False)
fastfood_sales_clean.to_csv(r'C:\Users\rougs\Desktop\data cleaning project\fastfoodclean.csv',index=False)

## Summary 

----------------------------------
Here is the summary of the cleaning phase

**DF2(Cafe, Post imputation, Pre-quarantine)**
|Column|Nulls|Nulls(AFTER)|
|------|-----|-----------|
|Transaction_ID |      0| 0|
|Item             |  333|501|
|Quantity         |  479|23|
|Price             | 533|6|
|Total_spent       | 502|23|
|Payment Method    |2579|2579|
|Order method     | 3265|3265|
|date              | 460|460|

This cleaning stage managed to recover a total of 135 Item,456 Quantity, 527 Price, and 479 Total_spent values. Note that for the Item column, the original null count is 636 including the 'ERROR' and 'UNKNOWN' count. 

Now these datasets are clean and complete enough to be analyzed in the next phase.



